In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity as ssim_metric
from skimage.metrics import peak_signal_noise_ratio as psnr_metric
import random
import math

try:
    from IPython.display import display, clear_output
except ImportError:
    pass

CONFIG = {
    "VOLUME_PATH": "sliepner.npy",
    "DEVICE": "cuda" if torch.cuda.is_available() else "cpu",
    "BATCH_SIZE": 16,           
    "EPOCHS": 50,               
    "LR": 1e-4,
    "SCALE": 2,
    "N_RESBLOCKS": 16,          
    "N_FEATS": 64,              
    "SAVE_DIR": "checkpoints",
    "PLOT_TRANSPOSE": True,
    "SEED": 42,
    "RESUME_TRAINING": True,    
    "CLIP_PERCENTILE": 1.0      # Percentile to clip for visualization
}

os.makedirs(CONFIG["SAVE_DIR"], exist_ok=True)
torch.manual_seed(CONFIG["SEED"])
np.random.seed(CONFIG["SEED"])
random.seed(CONFIG["SEED"])

def to_np(x):
    return x.detach().cpu().numpy()

def transpose_and_plot(ax, img, title):
    if isinstance(img, torch.Tensor):
        arr = img.squeeze().detach().cpu().numpy()
    else:
        arr = np.array(img)
    if CONFIG["PLOT_TRANSPOSE"]:
        arr = arr.T
    
    # Balanced Clipping for Seismic: Center on 0
    # We calculate the robust max amplitude based on the config percentile
    # e.g., if CLIP_PERCENTILE is 1.0, we look at the 99th percentile of absolute values
    robust_max = np.percentile(np.abs(arr), 100 - CONFIG["CLIP_PERCENTILE"])
    
    # Force symmetric limits for RdBu so 0 is always White/Gray
    ax.imshow(arr, cmap="RdBu", aspect="auto", vmin=-robust_max, vmax=robust_max)
    ax.set_title(title)
    ax.axis("off")

def seismic_degrade(hr, scale=CONFIG["SCALE"], sigma_blur=2.0, noise_std=0.02):
    B, C, H, W = hr.shape
    kernel_size = int(2 * math.ceil(2 * sigma_blur) + 1)
    if kernel_size % 2 == 0:
        kernel_size += 1
        
    coords = torch.arange(kernel_size, dtype=torch.float32, device=hr.device) - (kernel_size - 1) / 2.0
    g = torch.exp(-(coords**2) / (2 * sigma_blur**2))
    g = g / g.sum()
    
    g2d = g.view(-1, 1) @ g.view(1, -1) 
    g2d = g2d / g2d.sum()
    g2d = g2d.to(hr.device)
    g2d = g2d.view(1, 1, kernel_size, kernel_size)
    
    pad = kernel_size // 2
    blurred = F.conv2d(hr, g2d.repeat(C, 1, 1, 1), padding=pad, groups=C)
    
    small = F.interpolate(blurred, scale_factor=1/scale, mode="bilinear", align_corners=False)
    
    noise = noise_std * torch.randn_like(small)
    small = small + noise
    
    lr_up = F.interpolate(small, size=(H, W), mode="bilinear", align_corners=False)
    
    return lr_up, small

class SeismicSRDataset(Dataset):
    def __init__(self, volume_path, scale=CONFIG["SCALE"]):
        if not os.path.exists(volume_path):
            print(f"Dataset not found at {volume_path}, generating synthetic data.")
            vol = np.random.randn(100, 128, 128).astype(np.float32)
        else:
            vol = np.load(volume_path).astype(np.float32)
        
        # --- NEW NORMALIZATION STRATEGY ---
        # Previous min-max scaling shifted the zero crossing, causing "blue dominance".
        # Seismic data must be centered at 0.
        # We use a robust max scaling: divide by the 99.5th percentile of absolute amplitude.
        # This keeps 0 at 0 and handles outliers.
        
        self.norm_scale = np.percentile(np.abs(vol), 99.5)
        if self.norm_scale < 1e-6: self.norm_scale = 1.0 # Prevent div by zero
        
        print(f"Global Normalization Scale (Abs 99.5%): {self.norm_scale:.4f}")
        
        vol = vol / self.norm_scale
        # Clip strictly to [-1, 1] to handle extreme outliers outside the 99.5%
        vol = np.clip(vol, -1.0, 1.0)
        
        self.slices = [vol[i] for i in range(vol.shape[0])]
        self.scale = scale

    def __len__(self):
        return len(self.slices)

    def __getitem__(self, idx):
        hr = self.slices[idx]
        hr = torch.tensor(hr, dtype=torch.float32).unsqueeze(0)
        hr = hr.unsqueeze(0)
        lr_up, lr_small = seismic_degrade(hr, scale=self.scale)
        return lr_up.squeeze(0), hr.squeeze(0), lr_small.squeeze(0)

# ---------------------------------------------------------
# EDSR Architecture
# ---------------------------------------------------------

class ResBlock(nn.Module):
    def __init__(self, n_feats, kernel_size=3, bn=False, bias=True, act=nn.ReLU(True), res_scale=1):
        super(ResBlock, self).__init__()
        m = []
        for i in range(2):
            m.append(nn.Conv2d(n_feats, n_feats, kernel_size, padding=kernel_size//2, bias=bias))
            if bn: m.append(nn.BatchNorm2d(n_feats))
            if i == 0: m.append(act)

        self.body = nn.Sequential(*m)
        self.res_scale = res_scale

    def forward(self, x):
        res = self.body(x).mul(self.res_scale)
        res += x
        return res

class EDSR(nn.Module):
    def __init__(self, scale=CONFIG["SCALE"], n_resblocks=CONFIG["N_RESBLOCKS"], n_feats=CONFIG["N_FEATS"], n_colors=1):
        super(EDSR, self).__init__()
        kernel_size = 3
        act = nn.ReLU(True)
        self.head = nn.Conv2d(n_colors, n_feats, kernel_size, padding=kernel_size//2)
        m_body = [ResBlock(n_feats, kernel_size, act=act, res_scale=0.1) for _ in range(n_resblocks)]
        m_body.append(nn.Conv2d(n_feats, n_feats, kernel_size, padding=kernel_size//2))
        self.body = nn.Sequential(*m_body)
        m_tail = [
            nn.Conv2d(n_feats, n_feats * (scale ** 2), kernel_size, padding=kernel_size//2),
            nn.PixelShuffle(scale),
            nn.Conv2d(n_feats, n_colors, kernel_size, padding=kernel_size//2)
        ]
        self.tail = nn.Sequential(*m_tail)

    def forward(self, x):
        x = self.head(x)
        res = self.body(x)
        res += x
        x = self.tail(res)
        return x

def compute_metrics(sr, hr):
    sr_np = sr.detach().cpu().squeeze().numpy()
    hr_np = hr.detach().cpu().squeeze().numpy()
    # Metrics are computed on denormalized data now, so no clipping to [-1,1] required technically,
    # but PSNR typically expects a defined data range. 
    # For seismic, we treat the dynamic range as the range of the HR data.
    
    data_range = hr_np.max() - hr_np.min()
    if data_range == 0: data_range = 1.0

    if sr_np.ndim == 2:
        psnr_val = psnr_metric(hr_np, sr_np, data_range=data_range)
        ssim_val = ssim_metric(hr_np, sr_np, data_range=data_range)
        mae_val = np.mean(np.abs(sr_np - hr_np))
    else:
        psnr_val = 0; ssim_val = 0; mae_val = 0
        for i in range(sr_np.shape[0]):
             dr = hr_np[i].max() - hr_np[i].min()
             psnr_val += psnr_metric(hr_np[i], sr_np[i], data_range=dr)
             ssim_val += ssim_metric(hr_np[i], sr_np[i], data_range=dr)
             mae_val += np.mean(np.abs(sr_np[i] - hr_np[i]))
        psnr_val /= sr_np.shape[0]
        ssim_val /= sr_np.shape[0]
        mae_val /= sr_np.shape[0]
        
    return psnr_val, ssim_val, mae_val

def train():
    ds = SeismicSRDataset(CONFIG["VOLUME_PATH"], scale=CONFIG["SCALE"])
    loader = DataLoader(ds, batch_size=CONFIG["BATCH_SIZE"], shuffle=True, drop_last=True, num_workers=0)
    
    model = EDSR(scale=CONFIG["SCALE"]).to(CONFIG["DEVICE"])
    criterion = nn.L1Loss()
    opt = torch.optim.Adam(model.parameters(), lr=CONFIG["LR"])

    start_epoch = 1
    best_psnr = -1.0
    checkpoint_path = os.path.join(CONFIG["SAVE_DIR"], "edsr_best.pth")

    if CONFIG["RESUME_TRAINING"] and os.path.exists(checkpoint_path):
        print(f"Resuming training from {checkpoint_path}...")
        try:
            checkpoint = torch.load(checkpoint_path, map_location=CONFIG["DEVICE"], weights_only=False)
            model.load_state_dict(checkpoint['model'])
            opt.load_state_dict(checkpoint['opt'])
            start_epoch = checkpoint['epoch'] + 1
            best_psnr = checkpoint.get('best_psnr', -1.0)
            print(f"Resumed successfully. Starting at Epoch {start_epoch}, Best PSNR so far: {best_psnr:.4f}")
        except Exception as e:
            print(f"Failed to resume from checkpoint: {e}. Starting from scratch.")

    val_indices = sorted(list(set([min(5, len(ds)-1), min(50, len(ds)-1), len(ds)-1])))
    if len(ds) == 0: val_indices = []

    print(f"Starting EDSR training for {CONFIG['EPOCHS']} epochs (from epoch {start_epoch})...")
    
    for epoch in range(start_epoch, CONFIG["EPOCHS"]+1):
        model.train()
        pbar = tqdm(loader, desc=f"Epoch {epoch}/{CONFIG['EPOCHS']}")
        avg_loss = 0.0
        nb = 0

        for lr_up, hr, lr_small in pbar:
            lr_small = lr_small.to(CONFIG["DEVICE"])
            hr = hr.to(CONFIG["DEVICE"])
            sr = model(lr_small)
            if sr.shape != hr.shape:
                sr = F.interpolate(sr, size=hr.shape[2:], mode='bilinear', align_corners=False)
            loss = criterion(sr, hr)
            opt.zero_grad()
            loss.backward()
            opt.step()
            avg_loss += loss.item()
            nb += 1
            pbar.set_postfix({"loss": f"{loss.item():.6f}"})

        avg_loss_epoch = avg_loss / max(1, nb)
        print(f"Epoch {epoch} avg loss: {avg_loss_epoch:.6f}")

        # --- Validation ---
        model.eval()
        with torch.no_grad():
            try:
                clear_output(wait=True)
            except NameError:
                pass
            
            epoch_psnr_accum = 0.0
            
            for v_idx in val_indices:
                lr_up_val, hr_val, lr_small_val = ds[v_idx]
                
                lr_small_val = lr_small_val.unsqueeze(0).to(CONFIG["DEVICE"])
                hr_val = hr_val.unsqueeze(0).to(CONFIG["DEVICE"])
                lr_up_val = lr_up_val.unsqueeze(0)
                
                sr = model(lr_small_val)
                if sr.shape != hr_val.shape:
                    sr = F.interpolate(sr, size=hr_val.shape[2:], mode='bilinear', align_corners=False)
                
                sr = torch.clamp(sr, -1.0, 1.0)

                # --- DENORMALIZE FOR PLOTTING & METRICS ---
                # We multiply by the scaling factor to get back to original amplitudes
                sr_denorm = sr * ds.norm_scale
                hr_denorm = hr_val * ds.norm_scale
                lr_denorm = lr_up_val.to(CONFIG["DEVICE"]) * ds.norm_scale
                
                if lr_denorm.shape != hr_denorm.shape:
                     lr_denorm = F.interpolate(lr_denorm, size=hr_denorm.shape[2:], mode='bilinear', align_corners=False)
                
                psnr_v, ssim_v, mae_v = compute_metrics(sr_denorm, hr_denorm)
                psnr_b, ssim_b, mae_b = compute_metrics(lr_denorm, hr_denorm)
                
                epoch_psnr_accum += psnr_v

                try:
                    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
                    transpose_and_plot(axes[0], lr_denorm[0,0], f"Baseline (Bicubic)\nPSNR: {psnr_b:.2f} | MAE: {mae_b:.4f}")
                    transpose_and_plot(axes[1], hr_denorm[0,0], "HR (Original Denorm)")
                    transpose_and_plot(axes[2], sr_denorm[0,0], f"EDSR Output (Denorm)\nPSNR: {psnr_v:.2f} | MAE: {mae_v:.4f}")
                    plt.tight_layout()
                    plt.show() 
                except Exception as e:
                    pass

            avg_epoch_psnr = epoch_psnr_accum / max(1, len(val_indices))
            print(f"Epoch {epoch} Average Validation PSNR: {avg_epoch_psnr:.4f}")

            if avg_epoch_psnr > best_psnr:
                best_psnr = avg_epoch_psnr
                ckpt_path = os.path.join(CONFIG["SAVE_DIR"], "edsr_best.pth")
                torch.save({
                    "epoch": epoch, 
                    "model": model.state_dict(), 
                    "opt": opt.state_dict(),
                    "best_psnr": best_psnr
                }, ckpt_path)
                print(f"New best model saved with PSNR: {best_psnr:.4f} at {ckpt_path}")

    final_path = os.path.join(CONFIG["SAVE_DIR"], "edsr_final.pth")
    torch.save({"epoch": CONFIG["EPOCHS"], "model": model.state_dict(), "opt": opt.state_dict()}, final_path)

if __name__ == "__main__":
    train()